# Camillion — Single-Symbol Baseline (gravity only + 5m open-gate)

**The clean first experiment.** Edit only the **CONFIG** cell (mainly `SYMBOL`), then **Runtime -> Run all**. Gravity-only, one real 1m CSV, train/test split, one honest out-of-sample number. A 5m-CCI gate blocks new opens when the short-term market is flat.

In [ ]:
# ===== SETUP (run first) =====
REPO_URL = "https://github.com/monty313/Camillion_Claude_RL_model.git"
import os, sys, subprocess
REPO = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(REPO) and os.path.basename(os.getcwd()) != REPO:
    subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
subprocess.run(["git", "pull", "--quiet", "--ff-only"], check=False)
print("installing deps (~1-2 min first run)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "stable-baselines3[extra]", "gymnasium", "torch", "pandas", "numpy"], check=True)
from google.colab import drive
drive.mount("/content/drive")
print("setup done.")

In [ ]:
# ============================ EDIT ME (mainly SYMBOL) ============================
SYMBOL     = "EURUSD"                                   # <- your symbol
DRIVE_ROOT = "/content/drive/MyDrive/Camillion_data"    # one home for all 1m CSVs
CSV_PATH   = f"{DRIVE_ROOT}/{SYMBOL}_1m.csv"            # derived; override only if your filename differs
POSITION_SIZE   = 100000.0      # notional (tune per instrument)
REWARD_SCALE    = 1.0           # raise (e.g. 100-500) if per-step reward std is tiny
TOTAL_TIMESTEPS = 500_000       # use 20_000 for a quick smoke test FIRST
WARMUP          = 260           # bars skipped so 4h/1d indicators are warm
WINDOW          = 5_000         # bars per training episode (random window)
TRAIN_FRACTION  = 0.80          # first 80% train, last 20% out-of-sample test
OPEN_GATE_5M    = True          # block NEW opens when EITHER 5m CCI (30 or 100) is in [-50, 50]
SAVE_PATH       = f"/content/drive/MyDrive/camillion_single_{SYMBOL}"
# ================================================================================

In [ ]:
# ===== LOAD ONE CSV -> SPLIT -> TRAIN (gravity + 5m gate) -> OUT-OF-SAMPLE VERDICT =====
import numpy as np
from src.data.cache_builder import load_ohlcv_csv, build_aligned_indicators
from src.strategies.registry import AlphaRegistry
from src.strategies.gravity_30m_4h_alpha import register_gravity_alpha
from src.training.trainer import train, load_for_eval
from config.ftmo_config import load_active_config

def reg_factory():                       # GRAVITY ONLY
    r = AlphaRegistry(); register_gravity_alpha(r); return r

df = load_ohlcv_csv(CSV_PATH)
ind = build_aligned_indicators(df)
close = df["close"].values.astype("float32")
time_ns = df.index.values.astype("datetime64[ns]").astype("int64")
T = len(close); split = int(T * TRAIN_FRACTION)
print(f"{SYMBOL}: {T:,} 1m bars  ->  train {split:,} | test {T-split:,}  | 5m open-gate: {OPEN_GATE_5M}")

model = train(ind[:split], close[:split], time_ns[:split], reg_factory,
              total_timesteps=TOTAL_TIMESTEPS, n_envs=8, save_path=SAVE_PATH,
              position_size=POSITION_SIZE, reward_scale=REWARD_SCALE, warmup=WARMUP,
              window=WINDOW, random_window=True, open_gate=OPEN_GATE_5M)
print("trained; model + VecNormalize stats saved to", SAVE_PATH)

lo = max(0, split - WARMUP)
model, venv = load_for_eval(SAVE_PATH, ind[lo:], close[lo:], time_ns[lo:], reg_factory,
                            position_size=POSITION_SIZE, warmup=WARMUP, open_gate=OPEN_GATE_5M)
cfg = load_active_config(); start = cfg.starting_balance
obs = venv.reset(); eq = []
for _ in range(len(close[lo:]) - WARMUP - 2):
    a, _ = model.predict(obs, deterministic=True)
    obs, r, dones, infos = venv.step(a); eq.append(infos[0]["equity"])
    if bool(np.asarray(dones).any()):
        break
eq = np.array(eq, float)
if len(eq):
    ret = (eq[-1] - start) / start * 100
    peak = np.maximum.accumulate(eq); maxdd = ((peak - eq) / peak).max() * 100
    wall = float(getattr(cfg, "trailing_drawdown_pct", 4.0))
    print("=" * 56)
    print(f"  OUT-OF-SAMPLE {SYMBOL}:  return {ret:+.2f}%   maxDD {maxdd:.2f}%   bars {len(eq):,}")
    print(f"  4% wall -> {'BREACHED' if maxdd >= wall else 'survived'}")
    print("=" * 56)
    print("\nHonest gravity-only baseline (with 5m gate). Beat it by adding real alphas.")
else:
    print("no eval steps; check WARMUP vs test window size")